# ODE Integrators II: Implicit and Symplectic Methods

## Recap: Convergence and Stability

All integrators in these notes are one-step methods:
$x_{n+1}$ is computed from $x_n$ alone (possibly with intermediate
stages, as in Runge-Kutta).
A method is convergent of order $p$ if the global error
$e_N = |x_N - x(T)|$ at a fixed final time $T = N \Delta t$ scales as
$\mathcal{O}(\Delta t^p)$; this typically follows from a local
truncation error (the error of a single step) of
$\mathcal{O}(\Delta t^{p+1})$.
Every consistent one-step method converges as $\Delta t \to 0$
(for well-behaved $f$).

Convergence, however, does not tell us how large $\Delta t$ can be in
practice.
For that we study (absolute) stability with the linear test equation
\begin{align}
  \frac{dx}{dt} = \lambda x, \quad \lambda \in \mathbb{C},
  \quad \mathrm{Re}(\lambda) \leq 0,
\end{align}
whose exact solution does not grow.
A one-step method applied to it gives
\begin{align}
  x_{n+1} = R(z) x_n, \quad z = \lambda \Delta t,
\end{align}
where $R(z)$ is the amplification factor.
The method is stable for a given $z$ if $|R(z)| \leq 1$; the set of
such $z$ is the stability region.
A method whose stability region contains the entire left half-plane
$\mathrm{Re}(z) \leq 0$ is
[A-stable](https://en.wikipedia.org/wiki/Stiff_equation#A-stability).

For example, forward Euler $x_{n+1} = x_n + \Delta t f(x_n)$ is first
order with $R(z) = 1 + z$, so its stability region is the disk
$|1 + z| \leq 1$.
For real $\lambda < 0$ it is stable only if $\Delta t \leq 2/|\lambda|$.

We use two test problems below.

* Exponential decay:
  $dx/dt = -x$, i.e., the test equation with $\lambda = -1$, with
  solution $x(t) = e^{-t}$.

* Simple harmonic oscillator:
  \begin{align}
    \frac{d^2 x}{dt^2} = -\omega^2 x,
  \end{align}
  e.g., a mass $m$ on a spring of stiffness $k$, with $\omega^2 = k/m$.
  We realize it with a pendulum, whose full equation of motion
  $d^2\theta/dt^2 = -(g/l)\sin\theta$ is nonlinear but reduces to the
  harmonic oscillator with $x = \theta$ and $\omega^2 = g/l$ in the
  small-angle limit $\sin\theta \approx \theta$.
  Writing it as a first-order system with $\Omega = d\theta/dt$,
  \begin{align}
    \frac{d}{dt}
    \begin{bmatrix}
      \theta(t) \\
      \Omega(t)
    \end{bmatrix}
    =
    \begin{bmatrix}
      0    & 1 \\
      -g/l & 0
    \end{bmatrix}
    \begin{bmatrix}
      \theta(t) \\
      \Omega(t)
    \end{bmatrix},
  \end{align}
  where $\theta$ is the angular displacement, $\Omega$ the angular
  velocity, and $\omega = \sqrt{g/l}$ the natural frequency.
  The exact solution conserves the energy per unit mass
  $E = \tfrac{1}{2}(dx/dt)^2 + \tfrac{1}{2}\omega^2 x^2$,
  which for the pendulum reads
  $E = \tfrac{1}{2}\Omega^2 + \tfrac{1}{2}(g/l)\,\theta^2$.
  (Because $\theta$ is an angle rather than a length, the pendulum's
  physical energy is $m l^2 E$ rather than $m E$.)
  The matrix has purely imaginary eigenvalues $\lambda = \pm i\omega$,
  so the oscillator is the test equation with $z = \pm i\omega\Delta t$
  on the imaginary axis.

  

To see how the energy follows from $R$, diagonalize the oscillator
matrix
\begin{align}
  A =
  \begin{bmatrix}
    0         & 1 \\
    -\omega^2 & 0
  \end{bmatrix},
  \quad \omega^2 = g/l.
\end{align}

**Eigenvalues.**
The characteristic polynomial is
\begin{align}
  \det(A - \lambda I) =
  \begin{vmatrix}
    -\lambda  & 1 \\
    -\omega^2 & -\lambda
  \end{vmatrix}
  = \lambda^2 + \omega^2 = 0
  \quad \Rightarrow \quad
  \lambda_\pm = \pm i\omega.
\end{align}

**Eigenvectors.**
The first row of $(A - \lambda I)\,r = 0$ gives $r_2 = \lambda r_1$
(the second row, $-\omega^2 r_1 - \lambda r_2 = -(\omega^2 + \lambda^2) r_1 = 0$,
is then automatic).
Choosing $r_1 = 1$,
\begin{align}
  r_\pm =
  \begin{bmatrix}
    1 \\
    \pm i\omega
  \end{bmatrix}.
\end{align}
Collecting them as columns,
\begin{align}
  V =
  \begin{bmatrix}
    1        & 1 \\
    i\omega  & -i\omega
  \end{bmatrix},
  \quad
  V^{-1} = \frac{1}{2}
  \begin{bmatrix}
    1 & -i/\omega \\
    1 & i/\omega
  \end{bmatrix},
  \quad
  \Lambda =
  \begin{bmatrix}
    i\omega & 0 \\
    0       & -i\omega
  \end{bmatrix},
\end{align}
so that $A = V \Lambda V^{-1}$.
The rows of $V^{-1}$ are the left eigenvectors, $\ell A = \lambda \ell$.

**Diagonalizing the ODE.**
Write the state in the eigenvector basis, $x = V c$ with
$x = (\theta, \Omega)^T$ and $c = (c_+, c_-)^T$.
Substituting into $dx/dt = A x$ gives $V\,dc/dt = A V c$, and
multiplying by $V^{-1}$,
\begin{align}
  \frac{dc}{dt} = V^{-1} A V\, c = \Lambda c
  \quad \Leftrightarrow \quad
  \frac{dc_\pm}{dt} = \lambda_\pm c_\pm = \pm i\omega\, c_\pm,
  \quad
  c_\pm = \frac{1}{2}\left(\theta \mp \frac{i}{\omega}\Omega\right).
\end{align}
The new system is decoupled: each mode evolves independently as
$c_\pm(t) = c_\pm(0)\,e^{\pm i\omega t}$, and the original variables
are recovered from $x(t) = c_+(t)\,r_+ + c_-(t)\,r_-$.

Eigenvectors are defined only up to a factor.
Rescaling the modes as $u = 2\omega c_-$ and $\bar{u} = 2\omega c_+$
puts $\omega\theta$ and $\Omega$ on an equal footing:
\begin{align}
  \begin{bmatrix}
    u \\
    \bar{u}
  \end{bmatrix}
  =
  \begin{bmatrix}
    \omega & i \\
    \omega & -i
  \end{bmatrix}
  \begin{bmatrix}
    \theta \\
    \Omega
  \end{bmatrix},
  \quad
  \frac{d}{dt}
  \begin{bmatrix}
    u \\
    \bar{u}
  \end{bmatrix}
  =
  \begin{bmatrix}
    -i\omega & 0 \\
    0        & i\omega
  \end{bmatrix}
  \begin{bmatrix}
    u \\
    \bar{u}
  \end{bmatrix}.
\end{align}
The two equations are decoupled copies of the test equation, and for
real $\theta$ and $\Omega$ they are complex conjugates of each other,
so the complex amplitude $u = \omega\theta + i\Omega$ alone carries
the full state.
It satisfies $|u|^2 = \omega^2\theta^2 + \Omega^2 = 2E$.
A method with $u_{n+1} = R(z) u_n$ therefore gives
\begin{align}
  E_{n+1} = |R(z)|^2 E_n
  \quad \Rightarrow \quad
  E_n = |R(z)|^{2n} E_0,
\end{align}
where $|R(i\omega\Delta t)| = |R(-i\omega\Delta t)|$ for methods
with real coefficients.
The long-term behavior of a method is then set by $|R(z)|$ there:
* $|R| > 1$: artificial growth of energy.
  Forward Euler has $|R| = \sqrt{1 + \omega^2 \Delta t^2} > 1$ for
  every $\Delta t > 0$, so it is unconditionally unstable for the
  oscillator.
* $|R| < 1$: artificial damping of energy.
* $|R| = 1$: the amplitude is preserved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Grid in the complex z plane for plotting stability regions
Re = np.linspace(-3, 3, 601)
Im = np.linspace(-3, 3, 601)

Re, Im = np.meshgrid(Re, Im)
Z = Re + 1j * Im

# Forward Euler, allowing explicit time dependence in f(x, t)
def forwardEuler(f, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    for _ in range(n):
        X.append(X[-1] + dt * f(X[-1], T[-1]))
        T.append(T[-1] + dt)
    return np.array(X), np.array(T)

# Exponential decay: dx/dt = -x
f = lambda x, t: -x

# Simple harmonic oscillator
g, l = 9.8, 1

def Sho(x, t):
    theta, Omega = x
    return np.array([Omega, -(g/l) * theta])

def sho_solution(t):
    return np.cos(np.sqrt(g/l)*t)

## Explicit Methods

### Stability Regions for RK2 and RK4

Forward Euler is simple but has a very restrictive stability region.
In practice, physicists often use higher-order explicit Runge-Kutta
(RK) methods, such as RK2 and RK4.
These methods improve accuracy, and RK4 also noticeably enlarges the
stability region, including part of the imaginary axis.

To analyze stability, we apply the methods to the linear test equation:
\begin{align}
  \frac{dx}{dt} = \lambda x, \quad z = \lambda \Delta t.
\end{align}
Each Runge-Kutta scheme can be written in the form:
\begin{align}
  x_{n+1} = R(z) x_n,
\end{align}
where $R(z)$ is the amplification factor of the method.
Stability requires $|R(z)| \leq 1$.
* Forward Euler (for comparison):
  \begin{align}
    R_{\text{fE}}(z) = 1 + z
  \end{align}
* RK2 (explicit midpoint method):
  \begin{align}
    R_{\text{RK2}}(z) = 1 + z + \tfrac{1}{2} z^2
  \end{align}
* RK4 (classical Runge–Kutta):
  \begin{align}
    R_{\text{RK4}}(z) = 1 + z + \tfrac{1}{2} z^2 + \tfrac{1}{6} z^3 + \tfrac{1}{24} z^4
  \end{align}

Notice:
$R_{\text{RK4}}(z)$ is the Taylor expansion of $e^z$ through the
$z^4$ term (its first five terms), so it reproduces the exact
amplification $e^{\lambda \Delta t}$ with an error of
$\mathcal{O}(z^5)$.
This is consistent with RK4 being fourth-order accurate, although
matching $e^z$ on the linear test equation alone does not prove the
order for nonlinear problems.
Similarly, $R_{\text{RK2}}(z)$ matches $e^z$ through the $z^2$ term.

In [ ]:
# Amplification factors |R(z)|
RfE  = lambda z: abs(1 + z)  # for comparison only
RRK2 = lambda z: abs(1 + z + 0.5 * z**2)
RRK4 = lambda z: abs(1 + z + 0.5 * z**2 + (1/6) * z**3 + (1/24) * z**4)

# Plot regions
methods = {
    'Forward Euler': (RfE,  'C0'),
    'RK2':           (RRK2, 'C1'),
    'RK4':           (RRK4, 'C2'),
}

In [ ]:
def stable_regions(methods):
    for name, (R, color) in methods.items():
        plt.contour (Re, Im, R(Z), levels=[1],   colors=[color])
        plt.contourf(Re, Im, R(Z), levels=[0,1], colors=[color], alpha=0.1)

    plt.legend(handles=[
        mpatches.Patch(color=color, label=name) for name, (_,color) in methods.items()
    ])

    plt.xlabel(r"Re($z$) = Re($\lambda\Delta t$)")
    plt.ylabel(r"Im($z$) = Im($\lambda\Delta t$)")
    plt.gca().set_aspect('equal')

In [ ]:
stable_regions(methods)

Some observations from the above plot:
* Forward Euler:
  * Stability region:
    disk of radius 1 centered at $(-1,0)$.
  * Real axis: stable for $-2 \leq z \leq 0$.
  * Imaginary axis: unstable everywhere except $z = 0$.
* RK2
  * Stability region:
    larger than forward Euler's in the imaginary direction, but it
    reaches the same point $z = -2$ on the real axis
    ($R_{\text{RK2}}(-2) = 1$).
  * So for decaying problems with real $\lambda$, RK2 allows the same
    maximum time step as forward Euler, but with second-order
    accuracy.
  * Like forward Euler, it contains no part of the imaginary axis
    except $z = 0$, so it is still unstable for the undamped harmonic
    oscillator (although the growth is much slower).
* RK4
  * Stability region:
    extends to $z \approx -2.79$ on the real axis.
  * It includes part of the imaginary axis,
    $|\text{Im}(z)| \leq 2\sqrt{2} \approx 2.83$, so oscillatory
    systems are stable for $\omega \Delta t \lesssim 2.83$ (with
    $|R| < 1$, i.e., slight damping).
  * Most widely used in physics and engineering because of its balance
    between accuracy and stability.

In [ ]:
def RK2(f, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    for _ in range(n):
        k1 = f(X[-1],             T[-1]         )
        k2 = f(X[-1] + 0.5*dt*k1, T[-1] + 0.5*dt)
        X.append(X[-1] + dt * k2)
        T.append(T[-1] + dt     )
    return np.array(X), np.array(T)

In [ ]:
def RK4(f, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    for _ in range(n):
        k1 = f(X[-1]            , T[-1]         )
        k2 = f(X[-1] + 0.5*dt*k1, T[-1] + 0.5*dt)
        k3 = f(X[-1] + 0.5*dt*k2, T[-1] + 0.5*dt)
        k4 = f(X[-1] +     dt*k3, T[-1] +     dt)
        X.append(X[-1] + dt * (k1/6 + k2/3 + k3/3 + k4/6))
        T.append(T[-1] + dt)
    return np.array(X), np.array(T)

In [ ]:
t = 16
T = np.linspace(0, t, t*100+1)
X = np.exp(-T)

n = 9
X11, T11 = forwardEuler(f, x=1, t=0, dt=t/n, n=n)
X21, T21 = RK2         (f, x=1, t=0, dt=t/n, n=n)
X41, T41 = RK4         (f, x=1, t=0, dt=t/n, n=n)

n = 8
X12, T12 = forwardEuler(f, x=1, t=0, dt=t/n, n=n)
X22, T22 = RK2         (f, x=1, t=0, dt=t/n, n=n)
X42, T42 = RK4         (f, x=1, t=0, dt=t/n, n=n)

n = 6
X13, T13 = forwardEuler(f, x=1, t=0, dt=t/n, n=n)
X23, T23 = RK2         (f, x=1, t=0, dt=t/n, n=n)
X43, T43 = RK4         (f, x=1, t=0, dt=t/n, n=n)

n = 5
X14, T14 = forwardEuler(f, x=1, t=0, dt=t/n, n=n)
X24, T24 = RK2         (f, x=1, t=0, dt=t/n, n=n)
X44, T44 = RK4         (f, x=1, t=0, dt=t/n, n=n)

In [ ]:
plt.plot(T,   X,   'k',    label=r"$\exp(-t)$")

#plt.plot(T11, X11, "C0o-", label=f"Forward Euler with dt={t/9:.3f} (stable)")
plt.plot(T12, X12, "C1o-", label=f"Forward Euler with dt={t/8:.3f} (critical stability)")
#plt.plot(T13, X13, "C2o-", label=f"Forward Euler with dt={t/6:.3f} (unstable)")
#plt.plot(T14, X14, "C3o-", label=f"Forward Euler with dt={t/5:.3f} (unstable)")

#plt.plot(T21, X21, "C0^--", label=f"RK2 with dt={t/9:.3f} (stable)")
plt.plot(T22, X22, "C1^--", label=f"RK2 with dt={t/8:.3f} (critical stability)")
#plt.plot(T23, X23, "C2^--", label=f"RK2 with dt={t/6:.3f} (unstable)")
#plt.plot(T24, X24, "C3^--", label=f"RK2 with dt={t/5:.3f} (unstable)")

#plt.plot(T41, X41, "C0s:", label=f"RK4 with dt={t/9:.3f} (stable)")
plt.plot(T42, X42, "C1s:", label=f"RK4 with dt={t/8:.3f} (stable)")
#plt.plot(T43, X43, "C2s:", label=f"RK4 with dt={t/6:.3f} (stable)")
#plt.plot(T44, X44, "C3s:", label=f"RK4 with dt={t/5:.3f} (unstable)")

plt.xlabel(r"$t$")
plt.ylabel(r"$x(t)$")
plt.ylim(-10, 10)
plt.legend()

In [ ]:
# HANDSON*: comment and uncomment different curves to study them.


In general,
* Order vs. Stability:
  Higher order does not automatically mean a larger stability region
  (RK2 and forward Euler share the same real-axis limit $z = -2$), but
  RK4's region is substantially larger and includes part of the
  imaginary axis.
* Still Conditionally Stable:
  Neither RK2 nor RK4 is
  [A-stable](https://en.wikipedia.org/wiki/Stiff_equation#A-stability).
  They cannot remain stable for arbitrarily stiff problems where
  $\lambda$ is very negative (only in a relative sense!).
* Physics Implication:
  * For non-stiff problems (e.g. orbital dynamics, wave propagation),
    RK4 is an excellent choice.
  * For stiff problems (e.g. radiative cooling, chemical networks),
    even RK4 will fail unless $\Delta t$ is extremely small.
    These problems require implicit methods.

## Implicit Methods

So far we have focused on explicit methods such as forward Euler, RK2,
and RK4.
They are simple, intuitive, and accurate for many problems.
However, they have limitations:
* Forward Euler fails on oscillatory systems like the harmonic
  oscillator (unconditionally unstable).
* RK4 extends the stability region, but it is still
  conditionally stable:
  it cannot handle arbitrarily stiff problems.

### Example: A Stiff Problem in Astrochemistry

Consider H$_3^+$ formation in a dense molecular cloud.
Cosmic rays ionize H$_2$, the resulting H$_2^+$ is converted to
H$_3^+$ almost immediately, and H$_3^+$ is then slowly removed by
dissociative recombination:

| Reaction | Rate coefficient | Speed |
|---|---|---|
| H$_2$ + CR $\to$ H$_2^+$ + e$^-$ | $\zeta$ [s$^{-1}$] | slow |
| H$_2^+$ + H$_2$ $\to$ H$_3^+$ + H | $k_1$ [cm$^3$ s$^{-1}$] | fast |
| H$_3^+$ + e$^-$ $\to$ H$_2$ + H | $k_2$ [cm$^3$ s$^{-1}$] | slow at low $n_e$ |

We hold $n_{\mathrm{H_2}}$ fixed and assume charge balance,
$n_e = n_{\mathrm{H_2^+}} + n_{\mathrm{H_3^+}}$.
Writing $p = n_{\mathrm{H_2^+}}$ and $h = n_{\mathrm{H_3^+}}$,
\begin{align}
  \frac{dp}{dt} &= f_1 = \zeta n_{\mathrm{H_2}} - k_1 n_{\mathrm{H_2}} p, \\
  \frac{dh}{dt} &= f_2 = k_1 n_{\mathrm{H_2}} p - k_2 h (p + h).
\end{align}

To connect this nonlinear system to the linear test equation
$dx/dt = \lambda x$ used in our stability analysis, we linearize.
Perturb a solution, $\mathbf{x} = \mathbf{x}_0 + \delta\mathbf{x}$, and
Taylor-expand $\mathbf{f}$:
\begin{align}
  \frac{d\,\delta\mathbf{x}}{dt} \approx J\,\delta\mathbf{x}, \quad
  J = \frac{\partial \mathbf{f}}{\partial \mathbf{x}}.
\end{align}
Small errors therefore evolve according to the Jacobian $J$.
In the eigenbasis of $J$, each mode obeys its own test equation with
$\lambda$ equal to an eigenvalue $\lambda_i$ of $J$.
An explicit method is stable only if $z_i = \lambda_i \Delta t$ lies
inside its stability region for every $i$.
For a nonlinear system, $J$ depends on $\mathbf{x}$, so the
eigenvalues (and hence the stability limit) can change as the
solution evolves.

Here, with $\mathbf{x} = (p, h)$, the Jacobian is
\begin{align}
  J = \frac{\partial \mathbf{f}}{\partial \mathbf{x}} =
  \begin{bmatrix}
    \partial f_1/\partial p & \partial f_1/\partial h \\
    \partial f_2/\partial p & \partial f_2/\partial h
  \end{bmatrix}
  =
  \begin{bmatrix}
    -k_1 n_{\mathrm{H_2}}            & 0 \\
     k_1 n_{\mathrm{H_2}} - k_2 h    & -k_2 (p + 2h)
  \end{bmatrix}.
\end{align}
* The upper-right entry is zero because H$_2^+$ does not depend on
  H$_3^+$.
* The $-k_2 h$ in the lower-left entry appears because H$_2^+$ also
  contributes to $n_e$, which speeds up H$_3^+$ recombination.
* The $2h$ in the lower-right entry comes from the quadratic loss
  $k_2 h^2$.

Since $J$ is lower triangular, its eigenvalues are exactly the
diagonal entries:
\begin{align}
  \lambda_{\text{fast}} = -k_1 n_{\mathrm{H_2}}, \quad
  \lambda_{\text{slow}} = -k_2 (p + 2h).
\end{align}
* $\lambda_{\text{fast}} = -2\times10^{-5}$ s$^{-1}$ is constant.
  Its inverse, $1/(k_1 n_{\mathrm{H_2}}) = 5\times10^4$ s
  $\approx 14$ hours, is the (exact) e-folding time for H$_2^+$ to
  approach its equilibrium $\zeta/k_1 = 5\times10^{-9}$ cm$^{-3}$.
* $\lambda_{\text{slow}}$ changes as the solution evolves, because the
  system is nonlinear.
  It is 0 at $t = 0$ ($p = h = 0$).
  Near equilibrium, $h \approx \sqrt{\zeta n_{\mathrm{H_2}}/k_2} =
  10^{-3}$ cm$^{-3}$ (assuming $p \ll h$), so
  $\lambda_{\text{slow}} \approx -2\times10^{-10}$ s$^{-1}$.
  Its inverse, $1/(2 k_2 h) = 5\times10^9$ s $\approx 160$ years, is
  the e-folding time of H$_3^+$ near equilibrium.

The stiffness ratio near equilibrium is therefore
$|\lambda_{\text{fast}}/\lambda_{\text{slow}}| \sim 10^5$.
RK4 requires
$\Delta t < 2.79/|\lambda_{\text{fast}}| \approx 1.4\times10^5$ s
($\approx 1.6$ days), even though nothing interesting happens on
timescales shorter than centuries after the first few days.
This separation of timescales is what makes the problem stiff.

In [ ]:
zeta = 1e-17   # cosmic-ray ionization rate of H2 [s^-1]
k1   = 2e-9    # H2+ + H2 -> H3+ + H           [cm^3 s^-1]
k2   = 1e-7    # H3+ + e- -> products          [cm^3 s^-1]
nH2  = 1e4     # H2 number density             [cm^-3]
yr   = 3.156e7 # [s]

def h3p(x, t):
    nH2p, nH3p = x
    ne = nH2p + nH3p
    return np.array([zeta*nH2 - k1*nH2*nH2p,
                     k1*nH2*nH2p - k2*nH3p*ne])

lam  = -k1 * nH2
tend = 1000 * yr
print(f"lambda_fast = {lam:.1e} s^-1, RK4 limit dt < {2.79/abs(lam):.2e} s")
print(f"H3+ equilibrium n = {np.sqrt(zeta*nH2/k2):.1e} cm^-3")

# RK4 just below and just above its stability limit
dt_s, dt_u = 1.2e5, 1.5e5
Xs, Ts = RK4(h3p, x=[0,0], t=0, dt=dt_s, n=int(tend/dt_s))
with np.errstate(all='ignore'):
    Xu, Tu = RK4(h3p, x=[0,0], t=0, dt=dt_u, n=200)
print(f"RK4 with dt={dt_s:.1e} s: {len(Ts)-1} steps to reach {tend/yr:.0f} yr")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.loglog(Ts/yr, Xs[:,0], 'C0', label=r"H$_2^+$ (RK4, stable)")
ax.loglog(Ts/yr, Xs[:,1], 'C1', label=r"H$_3^+$ (RK4, stable)")
ax.axvline(1/abs(lam)/yr, color='gray', ls=':', label=r"$1/(k_1 n_{\rm H_2})$")
ax.set_xlabel(r"$t$ [yr]")
ax.set_ylabel(r"$n$ [cm$^{-3}$]")
ax.set_title(f"RK4, dt = {dt_s:.1e} s: {len(Ts)-1} steps")
ax.legend()

ax = axes[1]
ax.semilogy(Tu/yr*365.25, np.abs(Xu[:,0]), 'C3o-', ms=3, label=r"|H$_2^+$| (RK4, unstable)")
ax.axhline(zeta/k1, color='C0', ls='--', label=r"true H$_2^+$ equilibrium")
ax.set_xlabel(r"$t$ [days]")
ax.set_ylabel(r"$|n|$ [cm$^{-3}$]")
ax.set_title(f"RK4, dt = {dt_u:.1e} s: unstable")
ax.legend()

plt.tight_layout()

This raises a natural question:

> Can we design methods that remain stable even when
  $\lambda \Delta t$
  is very large and negative?

This leads us to implicit methods.

In explicit methods, the new state is computed directly from known
information,
\begin{align}
  x_{n+1} = \Phi(x_n, t_n, \Delta t).
\end{align}
An example is forward Euler $x_{n+1} = x_n + \Delta t f(x_n, t_n)$.
This makes them cheap and easy per step, but with limited stability.

In implicit methods, the new state appears on both sides of the
equation
\begin{align}
  x_{n+1} = \Psi(x_n, x_{n+1}, t_n, \Delta t)
\end{align}
Hence, an equation must be solved at each step:
a linear solve if $f$ is linear in $x$, or a root finder (e.g.,
Newton's method) if it is nonlinear.
An example is backward Euler $x_{n+1} = x_n + \Delta t f(x_{n+1},
t_{n+1})$.
This makes them more expensive per step, but they can have much
larger stability regions.
They are ideal for stiff systems.

### Backward Euler Method

The simplest implicit scheme is backward Euler, the implicit
counterpart to forward Euler.
\begin{align}
  x_{n+1} = x_n + \Delta t \, f(x_{n+1}, t_{n+1}).
\end{align}

This requires solving an equation for $x_{n+1}$ at every step.

For its stability analysis with the linear test equation, we apply
backward Euler to $dx/dt = \lambda x$ and define $z = \lambda \Delta
t$.
Given the update rule,
\begin{align}
  x_{n+1} = x_n + \Delta t \lambda x_{n+1}.
\end{align}

Rearrange:
\begin{align}
  x_{n+1} = \frac{1}{1 - z} x_n,
\end{align}
the amplification factor is:
\begin{align}
  R(z) = \frac{1}{1 - z}.
\end{align}

The stability condition requires
\begin{align}
  |R(z)| = \left|\frac{1}{1-z}\right| \leq 1.
\end{align}

* If $\text{Re}(z) \leq 0$ (stable in the continuous system), this
  inequality always holds.
* Therefore, backward Euler is
  [A-stable](https://en.wikipedia.org/wiki/Stiff_equation#A-stability):
  stable for the entire left half-plane.
* In fact, $|1/(1-z)| \leq 1$ means $|1 - z| \geq 1$:
  the stability region is the *exterior* of the disk of radius 1
  centered at $z = 1$.
  It even includes most of the right half-plane, where the exact
  solution grows.
* On the imaginary axis,
  $|R(i \omega \Delta t)| = 1/\sqrt{1 + \omega^2 \Delta t^2} < 1$,
  so backward Euler artificially damps oscillations such as the
  harmonic oscillator.

This is in stark contrast with forward Euler and explicit RK methods.

In [ ]:
# Amplification factors |R(z)|
RbE = lambda z: abs(1 / (abs(1-z) + 1e-308))  # trick to avoid divide by zero

# Add plot regions
methods['Backward Euler'] = (RbE, 'C3')

In [ ]:
stable_regions(methods)

Implementing the backward Euler method involves solving an implicit
equation at each time step.
The approach depends on whether the ODE is linear or nonlinear.

* Linear ODEs:
  If $f(x, t)$ is linear in $x$, the implicit equation can be solved
  directly without iterative methods.
  For example, for $dx/dt = \lambda x$:
  \begin{align}
    x_{n+1} = (1 - \lambda \Delta t)^{-1} x_n.
  \end{align}
  Using matrix inverse, the above formulation even works for system of
  linear ODEs.

* Nonlinear ODEs:
  If $f(x, t)$ is nonlinear, solving for $x_{n+1}$ typically requires
  iterative methods such as the Newton-Raphson method.
  Python's `fsolve()` from the `scipy.optimize` library can also be
  used for this purpose.

### Stiff ODEs

Let's implement the backward Euler method for a simple linear ODE and
compare it with the forward Euler method.

Consider:
\begin{align}
  \frac{dx}{dt} = -1000x - 2000e^{-t} + 3000, \quad x(0) = 0
\end{align}
The exact solution is:
\begin{align}
  x(t) = 3 - 0.998 e^{-1000t} - 2.002 e^{-t}
\end{align}
This ODE is stiff because it contains terms with vastly different
decay rates ($e^{-1000t}$ vs. $e^{-t}$).


In [ ]:
# Define the stiff ODE

def stiff(x, t):
    return -1000 * x - 2000 * np.exp(-t) + 3000

In [ ]:
# Define the exact solution for comparison

def stiff_solution(t):
    return 3 - (3-2000/999) * np.exp(-1000 * t) - (2000/999) * np.exp(-t)

In [ ]:
from scipy.optimize import fsolve

def backwardEuler(f, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    for _ in range(n):
        # Initial guess: Forward Euler estimate
        x0 = X[-1] + dt * f(X[-1], T[-1])
        
        # Define the implicit equation: x = x_last + dt * f(x, t)
        fim = lambda x: x - X[-1] - dt * f(x, T[-1] + dt)
        
        # Solve for x_next using Newton-Raphson (fsolve)
        # (reshape so that scalar and vector problems both work)
        x = fsolve(fim, x0).reshape(np.shape(X[-1]))
        
        X.append(x)
        T.append(T[-1] + dt)
    return np.array(X), np.array(T)

In [ ]:
# Parameters
x0 = 0
t0 = 0
tf = 0.01  # short time to observe stiffness effects
dt = 0.0025

T = np.linspace(t0, tf, 1000)
X = stiff_solution(T)

XfE, TfE = forwardEuler (stiff, x0, t0, dt, int(round((tf-t0)/dt)))
XbE, TbE = backwardEuler(stiff, x0, t0, dt, int(round((tf-t0)/dt)))

In [ ]:
# Plotting
plt.plot(T,   X,   "k-",   label="Exact Solution")
plt.plot(TfE, XfE, "C0o-", label="Forward Euler")
plt.plot(TbE, XbE, "C1o-", label="Backward Euler")
plt.xlabel(r"Time $t$")
plt.ylabel(r"$x(t)$")
plt.legend()

In [ ]:
# HANDSON: adjust dt and tf to observe the behaviors of forward and
#          backward Euler methods.
#          Specifically, try dt = 0.002, < 0.002, and > 0.002.


In [ ]:
# HANDSON*: apply backward Euler method to the simple harmonic
#          oscillator problem.
#          Plot evolution of energy as function of time.


#### Measuring the Order of Convergence

To measure the order $p$ of a method, we solve a problem with a known
solution up to a fixed final time $T$ for a sequence of step sizes, and
plot the global error $e_N = |x_N - x(T)|$ against $\Delta t$ on
log-log axes.
Since $e_N \approx C \Delta t^p$, the data fall on a line of slope $p$.
Here is an example for forward Euler on $dx/dt = -x$ with $T = 1$:

In [ ]:
fdecay = lambda x, t: -x
tend   = 1
Ns     = 2**np.arange(4, 12)

dts  = tend / Ns
errs = []
for N in Ns:
    XN, TN = forwardEuler(fdecay, x=1, t=0, dt=tend/N, n=N)
    errs.append(abs(XN[-1] - np.exp(-tend)))
errs = np.array(errs)

p = np.polyfit(np.log(dts), np.log(errs), 1)[0]
print(f"Measured order p = {p:.3f}")

plt.loglog(dts, errs, 'o-', label="Forward Euler")
plt.loglog(dts, errs[0] * (dts/dts[0]), 'k--', label="slope 1 reference")
plt.xlabel(r"$\Delta t$")
plt.ylabel(r"Global error $|x_N - x(T)|$")
plt.legend()

In [ ]:
# HANDSON*: create convergence plots to show the order of forward and
#          backward Euler method for both the stiff problem and
#          harmonic oscillator problem.


The comparison between forward Euler and backward Euler highlights
their complementary strengths and weaknesses:

* Forward Euler is simple and inexpensive per step, but it fails badly
  on stiff problems.
  Its step size is limited by the *fastest* mode
  ($\Delta t \leq 2/1000$ here), even long after that mode has decayed
  away and the solution only varies on the slow timescale $\sim 1$.
  This mismatch between the step size needed for accuracy and the one
  needed for stability is what makes a problem stiff.

* Backward Euler, in contrast, remains stable regardless of the step
  size for problems with $\text{Re}(\lambda) \leq 0$.
  This A-stability makes it ideally suited for stiff systems, where
  explicit schemes would require prohibitively small $\Delta t$.
  The price we pay is higher computational cost per step (solving
  implicit equations) as mentioned above, and potentially reduced
  accuracy.

In practice, implicit schemes are the method of choice whenever
stability is the dominant concern.
For example, in problems with a wide separation of timescales such as
chemical kinetics, radiative cooling, or diffusion processes.
They allow us to take much larger time steps while still capturing the
slow dynamics faithfully.

However, backward Euler introduces artificial damping in oscillatory
systems, suppressing dynamics that should remain bounded.
This motivates the search for implicit methods that retain the
stability benefits of backward Euler but reduce the damping error.
One elegant solution is the implicit midpoint method (equivalent to
the Crank–Nicolson, or trapezoidal, method for linear problems), which
we will study next.

### Implicit Midpoint Method

The implicit midpoint method is another implicit integrator.
It offers second-order accuracy and is symplectic (see below).
This makes it especially valuable for problems with an underlying
Hamiltonian structure, such as oscillators and orbital dynamics, where
energy conservation is critical over long simulations.
It can be viewed as the midpoint analogue of backward Euler: instead
of evaluating the derivative at the end of the interval, it evaluates
it at the midpoint between $x_n$ and $x_{n+1}$.

For the ODE
\begin{align}
  \frac{dx}{dt} = f(x,t), \quad x(t_0) = x_0,
\end{align}
the implicit midpoint update is
\begin{align}
  x_{n+1} = x_n + \Delta t f \left(\frac{x_n + x_{n+1}}{2}, t_n + \frac{\Delta t}{2}\right).
\end{align}
The derivative is evaluated at the midpoint of the interval
$[t_n, t_{n+1}]$.
This improves accuracy to second order, compared with the first-order
backward Euler scheme.

Apply to the linear test equation $dx/dt = \lambda x$:
\begin{align}
  x_{n+1} = x_n + \Delta t \lambda \left(\frac{x_n + x_{n+1}}{2}\right).
\end{align}

Rearranging:
\begin{align}
  \left(1 - \frac{z}{2}\right)x_{n+1} = \left(1 + \frac{z}{2}\right)x_n, \quad z = \lambda \Delta t.
\end{align}

Thus,
\begin{align}
  x_{n+1} = \frac{1 + z/2}{1 - z/2} \, x_n,
\end{align}
with amplification factor
\begin{align}
  R(z) = \frac{1 + z/2}{1 - z/2}.
\end{align}


* If $\text{Re}(z) \leq 0$, then $|R(z)| \leq 1$.
* Therefore, the implicit midpoint method is A-stable, like
  backward Euler.
* On the imaginary axis, $|R(i \omega \Delta t)| = 1$ exactly.
  So unlike backward Euler, it does not introduce artificial damping in
  oscillatory systems: it preserves the amplitude of oscillations.
* Caveat: as $z \to -\infty$, $R(z) \to -1$.
  Very stiff modes are therefore not damped quickly;
  with a large $\Delta t$ they flip sign every step and decay only
  slowly.
  Backward Euler, with $R(z) \to 0$, damps them strongly (a property
  called L-stability).

This makes it stable for stiff problems (with the caveat above) and
faithful to oscillatory and Hamiltonian dynamics over long time
integrations.

In [ ]:
# Amplification factors |R(z)|
Rim = lambda z: abs(1 + 0.5*z) / (abs(1 - 0.5*z) + 2e-308)  # trick to avoid divide by zero

# Add plot regions
methods['Implicit Midpoint'] = (Rim,  'C4')

In [ ]:
stable_regions(methods)

Let's apply the implicit midpoint method to the same stiff ODE as
before:

In [ ]:
def implicit_midpoint(f, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    for _ in range(n):
        t = T[-1] + dt
        
        # Initial guess: Forward Euler estimate
        x0 = X[-1] + dt * f(X[-1], T[-1])

        # Define the implicit equation: x = x_last + dt * f(x_mid, t_mid)
        fim = lambda x: x - X[-1] - dt * f((X[-1]+x)/2, (T[-1]+t)/2)
                
        # Solve for x_next using Newton-Raphson (fsolve)
        # (reshape so that scalar and vector problems both work)
        x = fsolve(fim, x0).reshape(np.shape(X[-1]))
        
        X.append(x)
        T.append(t)
    return np.array(X), np.array(T)

In [ ]:
# Parameters
x0 = 0
t0 = 0
tf = 0.01  # short time to observe stiffness effects
dt = 0.0025

T = np.linspace(t0, tf, 1000)
X = stiff_solution(T)

XfE, TfE = forwardEuler (stiff, x0, t0, dt, int(round((tf-t0)/dt)))
XbE, TbE = backwardEuler(stiff, x0, t0, dt, int(round((tf-t0)/dt)))
Xim, Tim = implicit_midpoint(stiff, x0, t0, dt, int(round((tf-t0)/dt)))

In [ ]:
# Plotting
plt.plot(T,   X,   "k-",   label="Exact Solution")
plt.plot(TfE, XfE, "C0o-", label="Forward Euler")
plt.plot(TbE, XbE, "C1o-", label="Backward Euler")
plt.plot(Tim, Xim, "C2o:", label="Implicit Midpoint")
plt.xlabel(r"Time $t$")
plt.ylabel(r"$x(t)$")
plt.legend()

In [ ]:
# HANDSON: adjust dt and tf to observe the behaviors of forward Euler,
#          backward Euler, and implicit midpoint methods.
#          Specifically, try dt = 0.002, < 0.002, and > 0.002.
#          Also try a large dt (e.g., 0.05) with a longer tf (e.g., 1)
#          and compare backward Euler with implicit midpoint.

In [ ]:
# HANDSON*: apply implicit midpoint method to the simple harmonic
#          oscillator problem.
#          Plot evolution of energy as function of time.


In [ ]:
# HANDSON: create convergence plots to show the order of Euler and
#          implicit midpoint method for both the stiff problem and
#          harmonic oscillator problem.


From the analysis and experiments above,
* Forward Euler:
  unstable unless $\Delta t \leq 2/|\lambda|$, which is tiny for stiff
  problems.
* Backward Euler:
  stable for any $\Delta t$, but damps oscillations.
* Implicit midpoint:
  stable for any $\Delta t$, second-order, and preserves the amplitude
  of oscillations (try it on the harmonic oscillator in the hands-on
  above).

This balance of stability and geometric fidelity makes the implicit
midpoint method (equivalent to Crank-Nicolson for linear problems) a
popular scheme for both stiff problems and Hamiltonian systems.

However, solving an implicit equation every step is unnecessary
overhead for non-stiff Hamiltonian systems.
Are there explicit methods that take advantage of the Hamiltonian
structure?

## Symplectic Algorithms

When simulating physical systems, especially those derived from
Hamiltonian mechanics, it is not enough for a numerical method to be
accurate and stable in the short term.
We also want to preserve the geometric structure of the system to
ensure that invariants such as energy and momentum remain bounded and
that phase-space volume is preserved.

Symplectic integrators are designed precisely for this purpose.
They preserve the symplectic structure of Hamiltonian flows, which
guarantees long-term qualitative fidelity, i.e.,
bounded oscillations,
stable orbits, and
correct statistical properties.

### Hamiltonian Systems

For a Hamiltonian
\begin{align}
  H(q,p) = K(p) + V(q)
\end{align}
with
* $q$: generalized coordinates (e.g. position).
* $p$: conjugate momenta.
* $K(p)$: kinetic energy.
* $V(q)$: potential energy.

The system's dynamics are governed by Hamilton's equations:
\begin{align}
  \frac{dq}{dt} = \frac{\partial H}{\partial p}, \quad
  \frac{dp}{dt} = -\frac{\partial H}{\partial q}.
\end{align}

Examples of Hamiltonian system include:
* Simple harmonic oscillator:
  \begin{align}
  H(q,p) = \frac{p^2}{2m} + \frac{1}{2}k q^2
  \end{align}
* Kepler problem (planetary motion):
  \begin{align}
  H(q,p) = \frac{p^2}{2m} - \frac{GMm}{|q|}
  \end{align}

The simple harmonic oscillator introduced above,
$\dot\theta = \Omega$, $\dot\Omega = -(g/l)\theta$, is this system with
$q = \theta$, $p = \Omega$, $m = 1$, and $k = g/l$.
Its right-hand side function `Sho(x, t)` therefore returns
$(\partial H/\partial p, -\partial H/\partial q)$, which is exactly
what the symplectic integrators below need.

### Conservation Laws

Hamiltonian systems obey key invariants:
* Energy conservation:
  if $H$ is not explicitly time-dependent, $dH/dt = 0$.
* Momentum conservation:
  if $H$ does not depend on a coordinate $q_i$, the corresponding
  $p_i$ is conserved
  ([Noether's theorem](https://en.wikipedia.org/wiki/Noether%27s_theorem)).
* Phase-space volume preservation:
  Liouville's theorem states that Hamiltonian flows preserve volume in
  $(q,p)$ space.

### Why Symplectic Integrators?

* Standard methods (e.g. forward Euler, RK4) cause systematic drift
  in energy over long runs.
  For the harmonic oscillator this follows from $|R(z)|$ on the
  imaginary axis:
  methods with $|R| > 1$ (forward Euler, RK2) artificially inject
  energy into the system, while methods with $|R| < 1$ (backward
  Euler, and RK4 for small $\Delta t$) artificially remove it.
* Symplectic integrators keep the energy error bounded and
  oscillatory (for sufficiently small $\Delta t$), never drifting away
  permanently.
* This makes them especially valuable in:
  * Molecular dynamics (bounded energy in large ensembles).
  * Celestial mechanics (stable planetary orbits).
  * Long-term oscillator simulations.

### Symplectic Euler Method

The symplectic Euler method updates the coordinates and momenta in a
staggered fashion, always using the most recently updated value.
There are two variants of the symplectic Euler method:
1. Kick-drift (update $p$ first):
   \begin{align}
     p_{n+1} &= p_n - \Delta t \cdot \frac{\partial V}{\partial q}(q_{n}) \\
     q_{n+1} &= q_n + \Delta t \cdot \frac{\partial K}{\partial p}(p_{n+1})
   \end{align}
2. Drift-kick (update $q$ first):
   \begin{align}
     q_{n+1} &= q_n + \Delta t \cdot \frac{\partial K}{\partial p}(p_n) \\
     p_{n+1} &= p_n - \Delta t \cdot \frac{\partial V}{\partial q}(q_{n+1})
   \end{align}

For a separable Hamiltonian $H = K(p) + V(q)$, both variants are
explicit.
(For non-separable Hamiltonians, one of the two updates becomes
implicit.)
As the formulation suggests, they are both first-order accurate in
time, like forward Euler.
Nevertheless, unlike forward or backward Euler, they preserve the
symplectic structure and keep the energy error bounded.
For the harmonic oscillator they are stable for $\omega \Delta t < 2$,
whereas forward Euler is unstable for every $\Delta t$.

In [ ]:
# Symplectic Euler Method (Drift-Kick Variant)

def symplecticEuler(F, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    
    for _ in range(n):
        q0, p0 = X[-1]
        t0     = T[-1]
        
        Hp, _  = F(X[-1], t0)
        q      = q0 + dt * Hp
        t      = t0 + dt
        
        _, mHq = F([q, p0], t)
        p      = p0 + dt * mHq
        
        X.append([q, p])
        T.append(t)
    return np.array(X), np.array(T)

In [ ]:
# Parameters
g, l = 9.8, 1
t, n = 100, 10000

T     = np.linspace(0, t, t*100+1)
Theta = sho_solution(T)

X1, T1 = forwardEuler(Sho, x=[1.0, 0.0], t=0, dt=t/n, n=n)
Theta1, Omega1 = X1[:,0], X1[:,1]

X2, T2 = symplecticEuler(Sho, x=[1.0, 0.0], t=0, dt=t/n, n=n)
Theta2, Omega2 = X2[:,0], X2[:,1]

plt.plot(T,  Theta,  'k',  label=r"Exact $\theta(t)$")
plt.plot(T1, Theta1,       label=r"Forward Euler $\theta(t)$")
plt.plot(T2, Theta2, '--', label=r"Symplectic Euler $\theta(t)$")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.ylim(-10, 10)
plt.legend()

In [ ]:
# HANDSON: Adjust the above parameters to observe the behavior of
#          symplectic Euler.


In [ ]:
# HANDSON*: create convergence plots to show the order of symplectic
#          Euler method for the harmonic oscillator problem.


From the above plot, we observe:
* Forward Euler Method:
  * Behavior:
    Deviates significantly from the exact solution over time,
    exhibiting growing oscillations.
  * Reason:
    $|R| > 1$ on the imaginary axis, so energy is artificially injected
    every step.
* Symplectic Euler Method:
  * Behavior:
    Maintains bounded energy errors, resulting in stable, oscillatory
    trajectories that closely follow the exact solution.
  * Reason:
    Preservation of the symplectic structure ensures that energy
    errors do not systematically drift.


### Leapfrog (Verlet) Integrator

The Leapfrog (or Verlet) method is one of the most widely used
symplectic integrators.
It improves upon symplectic Euler by achieving second-order accuracy
while still being explicit, symplectic, and time-reversible.

The key idea: positions $q$ and momenta $p$ are evaluated at staggered
time points, so that they "leapfrog" over each other.
This staggering makes the method symplectic and time-reversible, which
gives a bounded energy error even over very long simulations (for
$\omega \Delta t < 2$).

Leapfrog is especially important in molecular dynamics, celestial
mechanics, and $n$-body simulations, where millions of steps are
required to track conservative systems.

Start with Hamilton's equations:
\begin{align}
  \dot q = \frac{p}{m}, \qquad \dot p = -\frac{\partial V}{\partial q}(q).
\end{align}

Momenta are updated on half steps:
\begin{align}
  p_{n+1/2} = p_{n-1/2} - \frac{\partial V}{\partial q}(q_n)\,\Delta t,
\end{align}
then positions are updated on integer steps:
\begin{align}
  q_{n+1} = q_n + \frac{1}{m} p_{n+1/2}\,\Delta t.
\end{align}

Thus $p$ "leapfrogs" over $q$, each using the other's most up-to-date
value.

#### Synchronized (velocity-Verlet) Form

To avoid half-step bookkeeping, we can write Leapfrog in a
synchronized form:
\begin{align}
  q_{n+1} &= q_n + \frac{1}{m}p_n \,\Delta t - \tfrac{1}{2m} \frac{\partial V}{\partial q}(q_n) \Delta t^2, \\
  p_{n+1} &= p_n - \tfrac{1}{2}\left(\frac{\partial V}{\partial q}(q_n) + \frac{\partial V}{\partial q}(q_{n+1})\right)\Delta t.
\end{align}

This form is known as velocity-Verlet, and is the most widely used in
practice.

#### Kick-Drift-Kick Form

Another equivalent rearrangement is the **kick-drift-kick** scheme:
\begin{align}
  p_{n+1/2} &= p_n - \tfrac{1}{2} \frac{\partial V}{\partial q}(q_n)\,\Delta t, \\
  q_{n+1}   &= q_n + \frac{1}{m}p_{n+1/2}\,\Delta t, \\
  p_{n+1}   &= p_{n+1/2} - \tfrac{1}{2} \frac{\partial V}{\partial q}(q_{n+1})\,\Delta t.
\end{align}
Like velocity-Verlet, it starts and ends each step with $q$ and $p$ at
the same time, so it needs no special start-up step.
Since $\partial V/\partial q$ at $q_{n+1}$ from the end of one step can
be reused at the start of the next, only one new evaluation of
$\partial V/\partial q$ is needed per step.

In [ ]:
# Leapfrog Method (Kick-Drift-Kick Form)

def leapfrog(F, x, t, dt, n):
    X = [np.array(x)]
    T = [np.array(t)]
    
    for _ in range(n):
        q0, p0 = X[-1]
        t0     = T[-1]

        _, mHq = F(X[-1], t0)
        ph     = p0 + 0.5*dt * mHq
        th     = t0 + 0.5*dt
        
        Hp, _  = F([q0, ph], th)
        q      = q0 + dt * Hp
        t      = t0 + dt
        
        _, mHq = F([q, ph], t)
        p      = ph + 0.5*dt * mHq
        
        X.append([q, p])
        T.append(t)
    return np.array(X), np.array(T)

In [ ]:
# Parameters
t, n = 100, 10000

T     = np.linspace(0, t, t*100+1)
Theta = sho_solution(T)

X1, T1 = forwardEuler(Sho, x=[1.0, 0.0], t=0, dt=t/n, n=n)
Theta1, Omega1 = X1[:,0], X1[:,1]

X2, T2 = symplecticEuler(Sho, x=[1.0, 0.0], t=0, dt=t/n, n=n)
Theta2, Omega2 = X2[:,0], X2[:,1]

X3, T3 = leapfrog(Sho, x=[1.0, 0.0], t=0, dt=t/n, n=n)
Theta3, Omega3 = X3[:,0], X3[:,1]

plt.plot(T,  Theta,  'k',  label=r"Exact $\theta(t)$")
plt.plot(T1, Theta1,       label=r"Forward Euler $\theta(t)$")
plt.plot(T2, Theta2, '--', label=r"Symplectic Euler $\theta(t)$")
plt.plot(T3, Theta3, ':',  label=r"leapfrog $\theta(t)$")
plt.xlabel(r"$t$")
plt.ylabel(r"$\theta(t)$")
plt.ylim(-10, 10)
plt.legend()

In [ ]:
# HANDSON: Adjust the above parameters to observe the behavior of
#          leapfrog.


In [ ]:
# HANDSON: create convergence plots to show the order of symplectic
#          Euler and leapfrog method for the harmonic oscillator
#          problem.


In [ ]:
# HANDSON: use autodiff (e.g., JAX) to compute $\partial H/\partial q$
#          and $\partial H/\partial p$.


The Leapfrog integrator is:
* Accuracy:
  Second-order in $\Delta t$ (vs. first-order for symplectic Euler).
* Symplectic:
  Preserves the Hamiltonian structure, ensuring bounded energy error.
* Time-reversible:
  Running with $+\Delta t$ then $-\Delta t$ recovers the initial state
  exactly (up to round-off).
* Stability:
  Stable for oscillatory motion provided $\omega \Delta t < 2$
  (where $\omega$ is the oscillator frequency), the same as
  symplectic Euler.

### Derivation and Preservation of Symplectic Structure

So far, we have observed that symplectic integrators behave
differently:
they keep the energy error bounded,
preserve phase-space volume, and
do not drift over long times.
But *why* does this happen?

The answer lies in their ability to preserve the symplectic structure
of Hamiltonian systems.
This structure underlies Liouville's theorem (phase-space volume
preservation), and preserving it is what keeps the energy error of
symplectic integrators bounded.
Understanding how an algorithm preserves this structure explains why
symplectic methods are uniquely powerful.

For a separable Hamiltonian
\begin{align}
  H(q,p) = \frac{p^2}{2m} + V(q),
\end{align}
Hamilton's equations are
\begin{align}
  \dot q = \frac{\partial H}{\partial p} = \frac{p}{m},
  \qquad
  \dot p = -\frac{\partial H}{\partial q} = -V'(q).
\end{align}

A one-step method can be written as a map
\begin{align}
  (q_n, p_n) \mapsto (q_{n+1}, p_{n+1}).
\end{align}

For symplectic Euler:
\begin{align}
  q_{n+1} &= q_n + \tfrac{\Delta t}{m} p_n, \\
  p_{n+1} &= p_n - \Delta t \, V'(q_{n+1}).
\end{align}

The Jacobian of this transformation is
\begin{align}
  M =
  \begin{pmatrix}
  \frac{\partial q_{n+1}}{\partial q_n} & \frac{\partial q_{n+1}}{\partial p_n} \\
  \frac{\partial p_{n+1}}{\partial q_n} & \frac{\partial p_{n+1}}{\partial p_n}
  \end{pmatrix}
  =
  \begin{pmatrix}
  1 & \tfrac{\Delta t}{m} \\
  - \Delta t \, V''(q_{n+1}) & 1 - \tfrac{\Delta t^2}{m} V''(q_{n+1})
  \end{pmatrix}.
\end{align}

A method is symplectic if this Jacobian preserves the canonical
2-form:
\begin{align}
  M^T J M = J,
  \quad \text{where}
  \quad J =
  \begin{pmatrix}
     0 & 1 \\
    -1 & 0
  \end{pmatrix}.
\end{align}
(We write $J$ for this matrix to avoid confusion with the angular
velocity $\Omega$ of the oscillator.)
For one degree of freedom, $M^T J M = \det(M)\, J$, so the condition
reduces to $\det M = 1$.
For symplectic Euler,
\begin{align}
  \det M = 1 \cdot \left(1 - \tfrac{\Delta t^2}{m} V''(q_{n+1})\right)
         - \tfrac{\Delta t}{m} \cdot \left(-\Delta t \, V''(q_{n+1})\right)
         = 1.
\end{align}
Thus symplectic Euler is indeed symplectic.

The geometric interpretations here are:
* Symplecticity means the numerical flow preserves the geometry of
  Hamiltonian mechanics.
  Phase-space volume is preserved exactly, and the errors in
  invariants such as energy remain bounded.
* Contrast with non-symplectic methods:
  Forward Euler and RK4 distort the phase-space geometry, leading to
  secular energy drift (growth for forward Euler, decay for RK4).
* Discrete Hamiltonian Flow:
  Symplectic integrators can be interpreted as approximating a nearby
  Hamiltonian system rather than the exact one.
  This explains why energy error oscillates but does not drift.

The Jacobian also explains the stability limit $\omega \Delta t < 2$.
For the harmonic oscillator, $V = \frac{1}{2} k q^2$, so
$V'' = k$ is constant and $M$ is the same matrix at every step:
\begin{align}
  M =
  \begin{pmatrix}
  1 & \tfrac{\Delta t}{m} \\
  -k \Delta t & 1 - \omega^2 \Delta t^2
  \end{pmatrix},
  \qquad \omega^2 = \frac{k}{m}.
\end{align}
Its eigenvalues play the role of the amplification factor $R$.
They satisfy the characteristic equation
\begin{align}
  \det(M - \mu I)
  = (1 - \mu)\left(1 - \omega^2 \Delta t^2 - \mu\right) +
    \omega^2 \Delta t^2
  = \mu^2 - \left(2 - \omega^2 \Delta t^2\right)\mu + 1 = 0,
\end{align}
so
\begin{align}
  \mu_\pm = 1 - \frac{\omega^2 \Delta t^2}{2}
    \pm i\omega\Delta t \sqrt{1 - \frac{\omega^2 \Delta t^2}{4}}.
\end{align}
* For $\omega \Delta t < 2$, the square root is real, so $\mu_\pm$ are
  complex conjugates with
  $|\mu_\pm|^2 = \left(1 - \frac{\omega^2 \Delta t^2}{2}\right)^2 +
  \omega^2 \Delta t^2 \left(1 - \frac{\omega^2 \Delta t^2}{4}\right) = 1$
  exactly.
  The amplitude neither grows nor decays; only the phase is in error,
  with rotation angle per step $\cos\theta = 1 - (\omega \Delta t)^2/2$.
* At $\omega \Delta t = 2$, the square root vanishes, giving a double
  root $\mu = -1$ and linear growth.
  For $\omega \Delta t > 2$, the square root is imaginary, so the roots
  are real with $\mu_+ \mu_- = 1$, and one has $|\mu| > 1$
  (exponential growth).
  Either way, the method is unstable.

This is also why symplectic Euler has no scalar $R(z)$ in the summary
table.
It cannot be applied to $dx/dt = \lambda x$, because it requires
splitting the state into $q$ and $p$ and treats the two differently.
For forward Euler, diagonalizing a linear system reduces it to scalar
problems, each amplified by $R(\Delta t\,\lambda_i)$; for symplectic
Euler, diagonalizing mixes $q$ and $p$, so this reduction fails and
the matrix $M$ takes the place of $R$.

The "nearby Hamiltonian" can be written down explicitly for
the harmonic oscillator.
Although $H$ is not conserved exactly, the drift-kick map above
exactly conserves
\begin{align}
  \tilde{H} = \frac{p^2}{2m} + \frac{1}{2} k q^2 +
            \frac{k \Delta t}{2m}\, q\, p.
\end{align}

**Why $\tilde{H}$ is conserved.**
First, define 
\begin{align}
  \tilde{H}_n = \frac{p_n^2}{2m} + \frac{1}{2} k q_n^2 +
            \frac{k \Delta t}{2m}\, q_n\, p_n.
\end{align}

Since $q_n + \frac{\Delta t}{m} p_n = q_{n+1}$ (the drift), the last
two terms combine into $\frac{1}{2} k\, q_n q_{n+1}$, so
\begin{align}
  \tilde{H}_n = \frac{p_n^2}{2m} + \frac{1}{2} k\, q_n q_{n+1},
\end{align}
i.e., the potential energy uses the product of two consecutive
positions instead of $q_n^2$.
Using the kick in the form $p_{n+1} + k \Delta t\, q_{n+1} = p_n$,
\begin{align}
  \tilde{H}_{n+1}
  &= \frac{p_{n+1}^2}{2m} + \frac{k \Delta t}{2m}\, q_{n+1} p_{n+1} +
     \frac{1}{2} k\, q_{n+1}^2 \\
  &= \frac{1}{2m}\, p_{n+1}\left(p_{n+1} + k \Delta t\, q_{n+1}\right) +
     \frac{1}{2} k\, q_{n+1}^2 \\
  &= \frac{1}{2m}\left(p_n - k \Delta t\, q_{n+1}\right) p_n +
     \frac{1}{2} k\, q_{n+1}^2 \\
  &= \frac{p_n^2}{2m} +
     \frac{1}{2} k\, q_{n+1}\left(q_{n+1} - \tfrac{\Delta t}{m} p_n\right) \\
  &= \frac{p_n^2}{2m} + \frac{1}{2} k\, q_{n+1} q_n
   = \tilde{H}_n.
\end{align}

**Phase-space picture.**
$\tilde{H}$ is a quadratic form in $(q, p)$.
Its level sets are ellipses if it is positive definite, i.e., if
$\left(\frac{k \Delta t}{2m}\right)^2 < 4 \cdot \frac{1}{2m} \cdot \frac{k}{2}$,
which is exactly $\omega \Delta t < 2$.
So the stability limit has a geometric meaning:
* For $\omega \Delta t < 2$, the numerical points $(q_n, p_n)$ lie
  exactly on the closed ellipse $\tilde{H} = \tilde{H}_0$, which is
  slightly tilted (because of the $qp$ term) relative to the exact
  ellipse $H = H_0$.
* At $\omega \Delta t = 2$, the ellipse degenerates into a pair of
  parallel lines, and the points alternate between them while moving
  outward (linear growth).
* For $\omega \Delta t > 2$, the level sets are hyperbolas, and the
  points escape to infinity (exponential growth).

This also clarifies which "amplitude" is preserved in the stability
analysis above.
It is not the amplitude of $q$ or $p$ separately, but the modulus of
the eigen-mode of $M$, a complex combination of $q$ and $p$ analogous
to $u = \omega\theta + i\Omega$ in the recap:
\begin{align}
  \tilde{u} = \omega \left(q + \frac{\Delta t}{2m}\, p\right) +
      \frac{i}{m} \sqrt{1 - \frac{\omega^2 \Delta t^2}{4}}\; p,
  \qquad
  \tilde{u}_{n+1} = \mu_-\, \tilde{u}_n,
  \qquad
  |\tilde{u}|^2 = \frac{2\tilde{H}}{m}.
\end{align}
As $\Delta t \to 0$, $\tilde{u} \to \omega q + i p/m$, recovering
$u$ and $|u|^2 = 2E$ (with $m = 1$).
Since $|\mu_-| = 1$, $|\tilde{u}|$ and hence $\tilde{H}$ are constant,
while $q$ and $p$ move around the tilted ellipse, all at the same
numerical frequency set by $\cos\theta = 1 - (\omega \Delta t)^2/2$.


**Size of the energy error.**
Along the orbit, $H - \tilde{H} = -\frac{k \Delta t}{2m}\, q\, p$.
For small $\omega \Delta t$, $q \approx A \cos\omega t$ and
$p \approx -m\omega A \sin\omega t$, so
$q p \approx -\frac{1}{2} m\omega A^2 \sin 2\omega t$ and, with
$H \approx \frac{1}{2} k A^2$,
\begin{align}
  \frac{H - \tilde{H}}{\tilde{H}}
  \approx \frac{\omega \Delta t}{2} \sin 2\omega t.
\end{align}
The energy therefore oscillates twice per period with relative
amplitude $\omega \Delta t / 2$, first order in $\Delta t$ as expected
for a first-order method, and never drifts.
For the symplectic Euler plot above ($\omega \Delta t = \sqrt{9.8}
\times 0.01 \approx 0.031$) this is about 1.6%, which you can check
by plotting the energy in the hands-on.

**Beyond the harmonic oscillator.**
For a general (nonlinear) $H$, backward error analysis shows that a
symplectic method of order $p$ nearly conserves a modified Hamiltonian
$\tilde{H} = H + \mathcal{O}(\Delta t^p)$, given as a series in
$\Delta t$; the energy error then stays $\mathcal{O}(\Delta t^p)$ over
exponentially long times $t \sim e^{c/\Delta t}$.
For drift-kick symplectic Euler the leading terms are
\begin{align}
  \tilde{H} = H + \frac{\Delta t}{2}
              \frac{\partial H}{\partial p}
              \frac{\partial H}{\partial q} + \mathcal{O}(\Delta t^2),
\end{align}
which reproduces the $qp$ term above, since
$\frac{\partial H}{\partial p} \frac{\partial H}{\partial q} = \frac{p}{m}\, k q$.
(The kick-drift variant has the opposite sign.)
Leapfrog is second order, so its $\tilde{H} = H + \mathcal{O}(\Delta t^2)$
and its energy error is much smaller.
Non-symplectic methods such as forward Euler and RK4 have no such
conserved $\tilde{H}$, which is why their energy drifts.

Some practical considerations for using symplectic integrators
include:
* Choosing $\Delta t$:
  Symplectic integrators do not necessarily allow larger steps than
  non-symplectic ones (leapfrog needs $\omega \Delta t < 2$, while RK4
  is stable up to $\omega \Delta t \approx 2.83$).
  Their advantage is that, at a given $\Delta t$, they can run for very
  long times without energy drift.
  Accuracy still requires $\Delta t$ small enough to resolve the
  dynamics.
* Implicit schemes:
  For non-separable Hamiltonians, or implicit symplectic schemes
  (e.g. implicit midpoint), nonlinear equations must be solved each
  step.
* Adaptive time-stepping:
  Standard symplectic schemes assume constant $\Delta t$.
  Adaptive step sizes break symplecticity, though specialized adaptive
  symplectic schemes exist.

## Summary

| Method            | Type      | Order | $R(z)$                          | A-stable | Symplectic | Harmonic oscillator              |
|-------------------|-----------|-------|---------------------------------|----------|------------|----------------------------------|
| Forward Euler     | explicit  | 1     | $1 + z$                         | no       | no         | energy grows for every $\Delta t$ |
| RK2               | explicit  | 2     | $1 + z + z^2/2$                 | no       | no         | energy grows slowly              |
| RK4               | explicit  | 4     | $e^z$ through $z^4$             | no       | no         | energy slowly damped ($\omega\Delta t \lesssim 2.83$) |
| Backward Euler    | implicit  | 1     | $1/(1 - z)$                     | yes      | no         | energy damped                    |
| Implicit midpoint | implicit  | 2     | $(1 + z/2)/(1 - z/2)$           | yes      | yes        | amplitude preserved              |
| Symplectic Euler  | explicit* | 1     | —                               | no       | yes        | bounded energy error ($\omega\Delta t < 2$) |
| Leapfrog/Verlet   | explicit* | 2     | —                               | no       | yes        | bounded energy error ($\omega\Delta t < 2$) |

\*Explicit for separable Hamiltonians $H = K(p) + V(q)$.

* Convergence:
  every consistent one-step method converges as $\Delta t \to 0$
  (for well-behaved $f$);
  a local truncation error of $\mathcal{O}(\Delta t^{p+1})$ gives a
  global error of $\mathcal{O}(\Delta t^p)$.
* Stability:
  convergence does not tell us which $\Delta t$ to use;
  the stability region $|R(z)| \leq 1$ does.
  Stiff problems (very negative $\lambda$) require A-stable, i.e.,
  implicit, methods.
* Oscillatory problems:
  $z = \pm i\omega\Delta t$ lies on the imaginary axis, and $|R|$ there
  decides whether a method gains energy, loses energy, or preserves
  the amplitude.
* Symplectic integrators preserve the symplectic two-form, ensuring
  phase-space volume conservation and bounded energy error.
  This explains their excellent long-term behavior for Hamiltonian
  systems like oscillators and planetary motion.
* Unlike standard explicit methods, symplectic schemes are not just
  about minimizing local error.
  They preserve the very geometry of Hamiltonian dynamics.
  Interesting connection to
  [discrete differential geometry](https://en.wikipedia.org/wiki/Discrete_differential_geometry).